# 29. 확대 개발셋에서 E5·BM25 결합 가중치 비교

**목적:** 28번의 2기업·10질문·동일 가중치 결과만으로 검색 방식을 확정하지 않고,
5기업의 서로 다른 2024 사업보고서와 질문 25개에서 **벡터:BM25 비율만** 바꿔 비교합니다.

**후보:** `10:0`, `7:3`, `5:5`, `3:7`, `0:10`. `10:0`은 E5 단독, `0:10`은 BM25 단독입니다.
이 비율은 순위 점수에 주는 영향력이며 최종 결과에 포함할 문서 개수 비율이 아닙니다.
청킹(B 제목 경계 + E5 토큰 기준 400/48), E5 모델·revision, Kiwi 품사,
검색별 후보 20개, RRF 상수 60, 기업·연도 필터를 고정합니다.

**범위:** 삼성전자·리노공업·NAVER·LG생활건강·HDC의 2024 사업보고서, 기업당 5문항.
각 질문은 지정 기업·보고서 **내에서만** 검색합니다. 기업/연도 식별, 전 기업 통합 검색,
다년도 비교, 구조화 필드 DB 검색, parent-child 청킹, reranker, LLM 답변은 이번 대상이 아닙니다.
5개 회사라는 수는 통계적 충분성을 보장하지 않습니다.

**정답:** 검색 결과를 보기 전에 정한 질문·원문 인용문을 사용합니다. 인용문은 검색어·임베딩 입력에 넣지 않습니다.
삼성전자·리노공업 10문항은 28번 개발 질문을 재사용하고, 추가 기업 15문항은 공식 IR 사업보고서에서 수집했습니다.
따라서 **전체가 개발셋**이며, 독립 holdout이나 실사용 로그가 아닙니다. 같은 보고서의 다른 근거를 모두 판정한 완전한 relevance 정답도 아닙니다.

**실행:** `skn25` 커널에서 위에서 아래로 실행하고 출력이 저장되도록 저장합니다.
이번에는 OpenDART만 호출하며 OpenAI/LLM 호출·과금과 Qwen GPU 설정 변경은 없습니다.
실행 전 `.env`의 유효한 `OPENDART_API_KEY`를 확인하세요. 키·요청 URL은 출력하지 않습니다.

**2026-10-05 추출 수정:** 원문에 이스케이프되지 않은 `&` 때문에 `R&D` 등의 문자가 누락되던 문제를 보정했습니다.
질문·정답·청킹·가중치 조건은 변경하지 않았습니다. 아래 저장 출력은 수정 전 실행 기록이므로,
파일을 다시 열고 커널을 재시작한 뒤 처음부터 실행하세요. 28번을 따로 실행할 필요는 없습니다.

## 1. 환경과 28번의 검증된 실험 함수 재사용

실험 간 중복을 줄이기 위해 28번에서 **명시한 함수 정의 5개만** 가져옵니다.
28번 전체 셀, API·모델 호출, 저장된 출력은 실행하지 않습니다. 28번을 먼저 실행할 필요도 없습니다.
재사용 함수 내용은 승인된 AST 해시와 실행 전에 비교합니다. 함수가 변경되면 중단하며, 출력 저장만 바뀐 것은 허용합니다.
이는 notebook 사이의 실험용 의존성으로만 허용되며 최종 운영 코드는 이 방식에 의존하지 않습니다.

In [1]:
from io import BytesIO
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone
import ast, hashlib, importlib.metadata, json, os, re, sqlite3, sys, zipfile
import xml.etree.ElementTree as ET

from bs4 import BeautifulSoup, Comment, NavigableString, Tag
from dotenv import load_dotenv
from IPython.display import display
from kiwipiepy import Kiwi
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter
from transformers import AutoModel, AutoTokenizer
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F

if Path(sys.prefix).name != 'skn25':
    raise RuntimeError('이 notebook은 conda skn25 커널에서만 실행하세요.')
PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / 'src/dart/client.py').is_file())
load_dotenv(PROJECT_ROOT / '.env')
API_KEY = os.getenv('OPENDART_API_KEY')
if not API_KEY:
    raise RuntimeError('.env에 유효한 OPENDART_API_KEY가 필요합니다.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from dart.client import _get_bytes, CORP_CODE_URL

source_path = PROJECT_ROOT / 'notebooks/v1/28_hybrid_retrieval_paired_comparison.ipynb'
source_notebook = json.loads(source_path.read_text(encoding='utf-8'))
needed = {'download_xml', 'extract_b_text', 'morph_terms', 'encode_texts', 'bm25_order'}
definitions = []
for cell in source_notebook['cells']:
    if cell['cell_type'] == 'code':
        definitions.extend(node for node in ast.parse(''.join(cell['source'])).body
                           if isinstance(node, ast.FunctionDef) and node.name in needed)
if len(definitions) != len(needed) or {node.name for node in definitions} != needed:
    raise RuntimeError('28번의 재사용 함수가 변경되었습니다. 의존성을 확인하세요.')
reused_function_sha256 = hashlib.sha256(ast.dump(ast.Module(body=definitions, type_ignores=[])).encode()).hexdigest()
EXPECTED_FUNCTION_SHA256 = '2883c91a526dfb81e1aefe9919d5d651c0436ec19805c89cf42ea54662761bd2'
if reused_function_sha256 != EXPECTED_FUNCTION_SHA256:
    raise RuntimeError('28번 함수 내용이 검토된 버전과 다릅니다. 재검토 전에는 실행하지 않습니다.')
exec(compile(ast.Module(body=definitions, type_ignores=[]), str(source_path), 'exec'), globals())

# XML의 일반 &, 기존 엔티티, CDATA를 추출 중 누락·변형하지 않는지 확인합니다.
for raw, expected in [
    (b'<D><P>Home Care & Daily Beauty / R&D</P></D>', 'Home Care & Daily Beauty / R&D'),
    (b'<D><P>R&amp;D / &#38; / &#x26;</P></D>', 'R&D / & / &'),
    (b'<D><P>&lt;value&gt; &apos; &quot;</P></D>', "<value> ' \""),
    (b'<D><P><![CDATA[R&D / &amp;]]></P></D>', 'R&D / &amp;'),
    (b'<D><!-- R&D --><P>R&amp;amp;D</P></D>', 'R&amp;D'),
]:
    assert extract_b_text(raw) == expected, 'XML 특수문자 보존 회귀 검사 실패'

MODEL_ID = 'intfloat/multilingual-e5-small'
MODEL_REVISION = '5697a65b0a002a92fe8c4fc9d495303ffff9c7d2'
CANDIDATE_DEPTH, RRF_K, EVAL_K = 20, 60, 5
WEIGHTS = {'10:0': 1.0, '7:3': 0.7, '5:5': 0.5, '3:7': 0.3, '0:10': 0.0}
compact = lambda text: re.sub(r'[\s|]+', '', text)
print('skn25 확인 완료 | 함수 정의만 재사용 | E5 CPU | LLM/API 임베딩 호출 없음')

/home/sms/anaconda3/envs/skn25/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


skn25 확인 완료 | 함수 정의만 재사용 | E5 CPU | LLM/API 임베딩 호출 없음


## 2. 기업·질문·정답 고정

기업은 공개 종목코드로 OpenDART 회사 목록에서 확인합니다. 기업 식별 정확도를 평가하는 셀이 아닙니다.
아래 질문에는 제품·수익원·고객 관리·외환위험·연구개발·계절성·표 주의사항 같은 리서치 질문을 포함합니다.
삼성전자 기존 문항 3개는 기본 사실 조회이므로 질문 유형별 결과도 따로 봅니다.

In [2]:
REPORTS = {
    '삼성전자': {'stock_code': '005930', 'year': 2024, 'rcept_no': '20250311001085'},
    '리노공업': {'stock_code': '058470', 'year': 2024, 'rcept_no': '20250318000229'},
    'NAVER': {'stock_code': '035420', 'year': 2024, 'submitted': '20250318'},
    'LG생활건강': {'stock_code': '051900', 'year': 2024, 'submitted': '20250317'},
    'HDC': {'stock_code': '012630', 'year': 2024, 'submitted': '20250318'},
}
CASES = [
    {'id':'S01','company':'삼성전자','topic':'기본 사실','question':'삼성전자가 주식을 일반에 공개한 시점은 언제인가?','quote':'1975년 6월 11일 기업공개를 실시하였습니다.'},
    {'id':'S02','company':'삼성전자','topic':'기본 사실','question':'삼성전자공업에서 지금의 회사명으로 바뀐 날짜는?','quote':'1984년 2월 28일 정기주주총회 결의에 의거하여 상호를'},
    {'id':'S03','company':'삼성전자','topic':'기본 사실','question':'삼성전자 본사의 소재지는 어디인가?','quote':'경기도 수원시 영통구 삼성로 129(매탄동)'},
    {'id':'S04','company':'삼성전자','topic':'사업·제품','question':'삼성전자 반도체 사업 부문은 어떤 제품을 주로 만드는가?','quote':'DS 부문 | DRAM, NAND Flash, 모바일AP 등'},
    {'id':'S05','company':'삼성전자','topic':'사업·제품','question':'삼성전자 전장 사업에 해당하는 하만의 주요 제품은?','quote':'Harman | 디지털 콕핏(Digital Cockpit), 카오디오, 포터블 스피커 등'},
    {'id':'R01','company':'리노공업','topic':'수익·영업','question':'리노공업은 납품 단가를 정할 때 고객 주문에서 무엇을 살피는가?','quote':'거래처별 공급수량 및 제품의 난이도에 따라 합리적으로 가격을 책정'},
    {'id':'R02','company':'리노공업','topic':'수익·영업','question':'리노공업의 해외 거래처 관리 방식은 무엇인가?','quote':'지역별 혹은 국가별로 현지 대리점을 두어 고객을 관리하고 있으며'},
    {'id':'R03','company':'리노공업','topic':'회계 근거','question':'리노공업의 공정가치로 측정되는 금융상품 관련 공시 근거를 찾아줘.','quote':'공정가치로 측정되는 금융상품에 대하여 공정가치 측정'},
    {'id':'R04','company':'리노공업','topic':'회계 근거','question':'리노공업 퇴직급여 표에서 확정급여채무와 함께 어떤 적립자산 항목을 확인할 수 있나?','quote':'확정급여채무의 현재가치 | 사외적립자산'},
    {'id':'R05','company':'리노공업','topic':'위험','question':'리노공업이 시장 환경에서 경계하는 원자재·에너지 관련 요인은?','quote':'국내외 원자재 가격과 유가의 불안정'},
    {'id':'N01','company':'NAVER','topic':'수익·영업','question':'네이버의 서치플랫폼은 어떤 광고 사업으로 매출을 만드는가?','quote':'서치플랫폼은 검색과 디스플레이 광고로 구성되어 있습니다.'},
    {'id':'N02','company':'NAVER','topic':'수익·영업','question':'네이버 쇼핑 사업에서 광고 외에 돈을 버는 경로는 무엇인가?','quote':'커머스는 커머스 광고(쇼핑 관련 검색과 디스플레이 광고), 중개 및 판매(수수료), 멤버십 매출로 구성되어 있습니다.'},
    {'id':'N03','company':'NAVER','topic':'사업·제품','question':'네이버는 기업 고객 사업을 강화하기 위해 AI와 B2B 조직을 어느 법인 중심으로 통합했나?','quote':'AI 기술 조직과 B2B 사업 조직들을 네이버클라우드를 중심으로 조직 통합'},
    {'id':'N04','company':'NAVER','topic':'위험','question':'네이버 연결회사가 해외 영업으로 노출되는 환율 위험은 주로 어느 통화와 관련되는가?','quote':'외환 위험, 특히 주로 미국달러화, 일본 엔화 및 유로화와 관련된 환율 변동 위험'},
    {'id':'N05','company':'NAVER','topic':'연구개발','question':'네이버 연결 연구개발비를 해석할 때 연구개발이 주된 업무인 종속기업의 비용을 어떻게 포함하는지 확인해줘.','quote':'연구개발활동이 주된 업무인 경우 해당 법인의 전체 영업비용을 포함하여 산정되었습니다.'},
    {'id':'L01','company':'LG생활건강','topic':'사업·제품','question':'LG생활건강의 핵심 사업은 어떤 세 영역으로 구성되는가?','quote':'핵심 사업은 Beauty(화장품), HDB(생활용품, Home Care & Daily Beauty), Refreshment(음료)으로 구분됩니다.'},
    {'id':'L02','company':'LG생활건강','topic':'사업·제품','question':'LG생활건강의 생활용품 사업에서 개인 위생 제품과 집안 관리 제품은 어떻게 구분되는가?','quote':'Daily Beauty는 샴푸, 바디워시, 치약처럼 개인의 미용과 위생을 목적으로 사용되는 제품'},
    {'id':'L03','company':'LG생활건강','topic':'수익·영업','question':'LG생활건강은 코카콜라 제품을 직접 개발하는가, 원액을 받아 제조·판매하는가?','quote':'TCCC로부터 원액을 구매해 국내에서 제조하여 판매하는 방식으로 매출이 발생'},
    {'id':'L04','company':'LG생활건강','topic':'수익·영업','question':'LG생활건강의 화장품과 음료 사업은 계절적 매출 변동을 어떻게 보완하는가?','quote':'더운 여름이 성수기인 Refreshment 사업과 춥고 건조한 계절이 성수기인 Beauty 사업은 상호 보완하여 매출의 연중 변동성을 축소'},
    {'id':'L05','company':'LG생활건강','topic':'연구개발','question':'LG생활건강의 해외 연구소는 어느 국가에 있으며 어떤 역량을 강화하는가?','quote':'미국, 중국, 일본 연구소를 통해 글로벌 R&D 네트워크를 구축'},
    {'id':'H01','company':'HDC','topic':'수익·영업','question':'HDC 지주회사 자체의 사업 수익원에는 무엇이 있는가?','quote':'투자, 브랜드수수료, 경영자문수수료, 배당수익, 임대수익 등 지주회사 사업을 영위'},
    {'id':'H02','company':'HDC','topic':'회계 근거','question':'HDC 사업부문별 매출 표를 연결 매출액과 비교하기 전에 내부거래 처리에서 무엇을 주의해야 하는가?','quote':'내부거래금액을 제거하기 전 금액으로 작성되었습니다.'},
    {'id':'H03','company':'HDC','topic':'사업·제품','question':'HDC현대이피가 기존 금속을 대체하는 소재 사업을 강화하기 위해 양수한 사업은 무엇인가?','quote':'SK케미칼 PPS(Poly-Phenylene Sulfide) 사업을 양수'},
    {'id':'H04','company':'HDC','topic':'수익·영업','question':'HDC아이파크몰은 백화점 외에 어떤 입점 사업자로 임대수익을 확보하는가?','quote':'외식, 병원, 은행 등 다양한 테넌트의 유치를 통해 꾸준한 임대수익을 창출'},
    {'id':'H05','company':'HDC','topic':'사업·제품','question':'HDC의 통영 발전 사업은 언제 상업운전을 시작했나?','quote':'2024년 10월 29일 상업운전을 시작'},
]
assert len(CASES) == 25 and len({case['id'] for case in CASES}) == 25
assert {case['company'] for case in CASES} == set(REPORTS)
assert all(sum(case['company'] == company for case in CASES) == 5 for company in REPORTS)
evaluation_sha256 = hashlib.sha256(json.dumps(CASES, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
display(pd.DataFrame(CASES)[['id', 'company', 'topic', 'question']])
print('질문·정답 고정 SHA256:', evaluation_sha256)

,id,company,topic,question
0,S01,삼성전자,기본 사실,삼성전자가 주식을 일반에 공개한 시점은 언제인가?
1,S02,삼성전자,기본 사실,삼성전자공업에서 지금의 회사명으로 바뀐 날짜는?
2,S03,삼성전자,기본 사실,삼성전자 본사의 소재지는 어디인가?
3,S04,삼성전자,사업·제품,삼성전자 반도체 사업 부문은 어떤 제품을 주로 만드는가?
4,S05,삼성전자,사업·제품,삼성전자 전장 사업에 해당하는 하만의 주요 제품은?
5,R01,리노공업,수익·영업,리노공업은 납품 단가를 정할 때 고객 주문에서 무엇을 살피는가?
6,R02,리노공업,수익·영업,리노공업의 해외 거래처 관리 방식은 무엇인가?
7,R03,리노공업,회계 근거,리노공업의 공정가치로 측정되는 금융상품 관련 공시 근거를 찾아줘.
8,R04,리노공업,회계 근거,리노공업 퇴직급여 표에서 확정급여채무와 함께 어떤 적립자산 항목을 확인할 수 있나?
9,R05,리노공업,위험,리노공업이 시장 환경에서 경계하는 원자재·에너지 관련 요인은?


질문·정답 고정 SHA256: 3005bdf10873856eb367da65ccebd7597edd6c230ec9ee7caeea101bd637cffd


## 3. 공식 OpenDART 보고서 선택과 B 텍스트 추출

추가 3기업의 접수번호는 공식 IR PDF의 제출일에 맞춰 `A001` 공시검색으로 찾습니다.
같은 날·기업에 정확한 `사업보고서 (2024.12)`가 하나가 아니면 임의 선택하지 않고 중단합니다.
삼성전자·리노공업은 28번과 동일한 접수번호를 유지하고 공식 목록에서 기업·보고서와의 관계도 대조합니다. **최신 정정보고서 자동 선택 실험은 아닙니다.**
본문 표지의 기업명·사업연도를 확인하고, 접수번호·원문 해시·XML·텍스트 용량을 기록합니다.
새 회사의 ZIP에서도 `<접수번호>.xml`을 명시적으로 선택하며 가장 큰 XML을 무조건 선택하지 않습니다.

In [3]:
def get_list_json(params):
    try:
        payload = _get_bytes('https://opendart.fss.or.kr/api/list.json',
                             {'crtfc_key': API_KEY, **params}, timeout=120)
        result = json.loads(payload)
    except Exception:
        raise RuntimeError('OpenDART 공시 목록 요청 실패. 연결·인증을 확인하세요. URL·키는 출력하지 않습니다.') from None
    if result.get('status') != '000':
        raise RuntimeError(f"OpenDART 목록 상태 {result.get('status', 'UNKNOWN')}; 조회 조건·인증을 확인하세요.")
    return result

try:
    corp_zip = _get_bytes(CORP_CODE_URL, {'crtfc_key': API_KEY}, timeout=120)
except Exception:
    raise RuntimeError('기업 목록 요청 실패. API 인증·연결을 확인하세요. URL은 출력하지 않습니다.') from None
if not zipfile.is_zipfile(BytesIO(corp_zip)):
    raise RuntimeError('기업 목록이 정상 ZIP이 아닙니다. API 상태를 확인하세요.')
with zipfile.ZipFile(BytesIO(corp_zip)) as archive:
    info = archive.getinfo('CORPCODE.xml')
    if info.file_size > 100_000_000:
        raise RuntimeError('기업 목록 압축 해제 한도 초과')
    root = ET.fromstring(archive.read(info))
source_texts, source_rows = {}, []
for company, report in REPORTS.items():
    matches = [item for item in root.findall('list') if item.findtext('stock_code') == report['stock_code']]
    if len(matches) != 1:
        raise RuntimeError(f'{company}: 종목코드에 대응하는 기업이 한 개가 아닙니다.')
    corp_code = matches[0].findtext('corp_code')
    if not re.fullmatch(r'\d{8}', corp_code or ''):
        raise RuntimeError(f'{company}: 기업코드 형식 오류')
    submitted = report.get('submitted') or report['rcept_no'][:8]
    result = get_list_json({'corp_code': corp_code, 'bgn_de': submitted, 'end_de': submitted,
                           'pblntf_detail_ty': 'A001', 'last_reprt_at': 'N',
                           'page_count': '100', 'page_no': '1'})
    if int(result.get('total_page', 0)) != 1 or int(result.get('total_count', -1)) != len(result.get('list', [])):
        raise RuntimeError(f'{company}: 보고서 조회 페이지·건수가 예상과 다릅니다.')
    choices = [item for item in result.get('list', [])
               if item.get('corp_code') == corp_code and item.get('rcept_dt') == submitted
               and re.sub(r'\s+', '', item.get('report_nm', '')) == '사업보고서(2024.12)'
               and ('rcept_no' not in report or item.get('rcept_no') == report['rcept_no'])]
    if len(choices) != 1:
        raise RuntimeError(f'{company}: 지정 제출일·접수번호의 2024 사업보고서가 한 개가 아닙니다.')
    report['rcept_no'] = choices[0]['rcept_no']
    receipt = report['rcept_no']
    if not re.fullmatch(r'\d{14}', receipt):
        raise RuntimeError('접수번호 형식 오류')
    print(f'{company} 원문 준비 시작 | 접수번호 {receipt}', flush=True)
    started = perf_counter()
    xml_bytes = download_xml(receipt, receipt + '.xml')
    text = extract_b_text(xml_bytes)
    header = compact(text[:6000])
    name_aliases = {'삼성전자': ('삼성전자',), '리노공업': ('리노공업',),
                    'NAVER': ('네이버', 'NAVER'), 'LG생활건강': ('LG생활건강', '엘지생활건강'),
                    'HDC': ('에이치디씨', 'HDC')}
    if not any(alias in header for alias in name_aliases[company]):
        raise RuntimeError(f'{company}: 본문 표지에서 기업명을 확인하지 못했습니다.')
    if not all(marker in header for marker in ('2024년01월01일', '2024년12월31일')):
        raise RuntimeError(f'{company}: 표지의 사업연도가 2024년인지 확인하지 못했습니다.')
    source_texts[company] = text
    source_rows.append({'company': company, 'corp_code': corp_code, 'business_year': 2024,
                        'receipt': receipt, 'xml_MB': len(xml_bytes)/1024**2,
                        'text_chars': len(text), 'text_utf8_MB': len(text.encode())/1024**2,
                        'source_sha256': hashlib.sha256(xml_bytes).hexdigest(),
                        'download_extract_seconds': perf_counter()-started,
                        'source_url': f'https://dart.fss.or.kr/dsaf001/main.do?rcpNo={receipt}'})
    print(f'{company} 준비 완료 | {len(text):,}자', flush=True)
display(pd.DataFrame(source_rows))
del corp_zip, root

삼성전자 원문 준비 시작 | 접수번호 20250311001085
삼성전자 준비 완료 | 44,808자
리노공업 원문 준비 시작 | 접수번호 20250318000229
리노공업 준비 완료 | 159,565자
NAVER 원문 준비 시작 | 접수번호 20250318000645
NAVER 준비 완료 | 638,895자
LG생활건강 원문 준비 시작 | 접수번호 20250317000790
LG생활건강 준비 완료 | 426,263자
HDC 원문 준비 시작 | 접수번호 20250318001151
HDC 준비 완료 | 62,014자


,company,corp_code,business_year,receipt,xml_MB,text_chars,text_utf8_MB,source_sha256,download_extract_seconds,source_url
0,삼성전자,00126380,2024,20250311001085,6.006890,44808,0.079376,b0682fd46de50ecc41f074bb43cb58cf574dd175034371...,0.329731,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
1,리노공업,00369657,2024,20250318000229,1.486798,159565,0.297894,a23af2dbcc2f4de4be59c5925ed45fceae84c23f4e78ff...,0.680538,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
2,NAVER,00266961,2024,20250318000645,6.900676,638895,1.078473,b1478f612c3ff07373d14fe9c7638936c77fcea2130aa4...,3.391410,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
3,LG생활건강,00356370,2024,20250317000790,4.573826,426263,0.764051,9ccc7d18e1ba66ce3982f41899d22d1dcb7ae8dfaf784f...,2.831921,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
4,HDC,00164636,2024,20250318001151,6.137928,62014,0.108966,53b41f2833b68f06bfea13a07abed1a7cdb3caf53c44e6...,0.406754,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...


## 4. 고정 청킹과 정답 근거 검사

E5 토크나이저 기준으로 제목 경계를 먼저 나누고 400/48로 청킹합니다.
인용문이 원문이나 청크에 없으면 그 문항을 조용히 제외해 점수를 높이지 않고 **전체 비교를 중단**합니다.
공백과 표 구분자 `|`만 무시하며 오타·단어를 임의로 고치지 않습니다.
같은 인용문이 겹침·중복 설명으로 여러 청크에 있으면 모두 알려진 정답 청크로 취급합니다.
이는 인용문 포함 여부를 보는 평가이며 관련 근거 전부를 판정한 정답 집합은 아닙니다.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
splitter = SentenceSplitter(chunk_size=400, chunk_overlap=48,
                           tokenizer=lambda text: tokenizer.encode(text, add_special_tokens=False))
chunks_by_company, chunk_rows = {}, []
for company, text in source_texts.items():
    sections = [part for part in re.split(r'(?=^## )', text, flags=re.MULTILINE) if part.strip()]
    chunks = [node.text.strip() for section in sections
              for node in splitter.get_nodes_from_documents([Document(text=section)]) if node.text.strip()]
    lengths = [len(tokenizer('passage: ' + chunk, truncation=False)['input_ids']) for chunk in chunks]
    if not chunks or max(lengths) > 512:
        raise RuntimeError(f'{company}: 빈 청크 또는 E5 입력 한도 초과. 임베딩을 중단합니다.')
    chunks_by_company[company] = chunks
    chunk_rows.append({'company': company, 'chunks': len(chunks), 'max_e5_tokens': max(lengths),
                       'over_512': sum(length > 512 for length in lengths)})

gold_indices, gold_rows = {}, []
for case in CASES:
    needle = compact(case['quote'])
    text = source_texts[case['company']]
    ids = [i for i, chunk in enumerate(chunks_by_company[case['company']]) if needle in compact(chunk)]
    gold_indices[case['id']] = set(ids)
    gold_rows.append({'id': case['id'], 'company': case['company'],
                      'source_occurrences': compact(text).count(needle), 'known_gold_chunks': len(ids)})
gold_audit = pd.DataFrame(gold_rows)
display(pd.DataFrame(chunk_rows))
display(gold_audit)
if (gold_audit['source_occurrences'] == 0).any() or (gold_audit['known_gold_chunks'] == 0).any():
    bad = gold_audit[(gold_audit['source_occurrences'] == 0) | (gold_audit['known_gold_chunks'] == 0)]
    raise RuntimeError(f"정답 근거가 없는 문항 {bad['id'].tolist()}. 원문·추출·청킹을 확인하고 비교는 중단하세요.")
print('모든 25문항의 원문·청크 근거 확인 완료 | 잘라낸 E5 입력 없음')

Token indices sequence length is longer than the specified maximum sequence length for this model (2734 > 512). Running this sequence through the model will result in indexing errors


,company,chunks,max_e5_tokens,over_512
0,삼성전자,82,403,0
1,리노공업,333,404,0
2,NAVER,1168,404,0
3,LG생활건강,823,404,0
4,HDC,122,404,0


,id,company,source_occurrences,known_gold_chunks
0,S01,삼성전자,1,1
1,S02,삼성전자,1,1
2,S03,삼성전자,4,4
3,S04,삼성전자,2,2
4,S05,삼성전자,1,1
5,R01,리노공업,1,1
6,R02,리노공업,1,1
7,R03,리노공업,1,1
8,R04,리노공업,4,4
9,R05,리노공업,2,2


RuntimeError: 정답 근거가 없는 문항 ['L01', 'L05']. 원문·추출·청킹을 확인하고 비교는 중단하세요.

## 5. BM25·E5 인덱스 준비 — 가중치별로 반복하지 않음

BM25는 Kiwi 형태소 + 메모리 SQLite FTS5입니다. XML 필드를 구조화한 SQL 조회는 아닙니다.
E5는 CPU에서 문서별 임베딩을 한 번 생성하고 가중치 후보 모두가 공유합니다.
문서·질문 묶음의 진행 상황과 인덱스 크기를 출력합니다. 모델 파일 크기·프로세스 메모리와 인덱스 배열 크기는 구분합니다.

In [ ]:
kiwi = Kiwi()
bm25_connections, index_rows = {}, []
for company, chunks in chunks_by_company.items():
    print(f'{company} BM25 준비 | {len(chunks)}청크', flush=True)
    started = perf_counter()
    terms = [' '.join(morph_terms(chunk)) for chunk in chunks]
    connection = sqlite3.connect(':memory:')
    connection.execute("CREATE VIRTUAL TABLE passage USING fts5(terms, content='')")
    connection.executemany('INSERT INTO passage(rowid, terms) VALUES (?, ?)',
                           [(i+1, value) for i, value in enumerate(terms)])
    connection.commit()
    bm25_connections[company] = connection
    size = connection.execute('PRAGMA page_count').fetchone()[0] * connection.execute('PRAGMA page_size').fetchone()[0]
    index_rows.append({'company': company, 'bm25_build_seconds': perf_counter()-started,
                       'bm25_sqlite_MiB': size/1024**2, 'empty_bm25_chunks': sum(not value for value in terms)})
display(pd.DataFrame(index_rows))

In [ ]:
torch.set_num_threads(min(4, os.cpu_count() or 1))
print('E5 CPU 모델 로드 시작', flush=True)
started = perf_counter()
model = AutoModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION, use_safetensors=True).to('cpu').eval()
model_load_seconds = perf_counter()-started
model_parameter_MiB = sum(p.numel()*p.element_size() for p in model.parameters())/1024**2
dense_index, dense_rows = {}, []
for company, chunks in chunks_by_company.items():
    print(f'{company} E5 임베딩 시작 | {len(chunks)}청크', flush=True)
    started = perf_counter()
    batches = []
    for offset in range(0, len(chunks), 64):
        batches.append(encode_texts(chunks[offset:offset+64], 'passage'))
        print(f"  {company} {min(offset+64,len(chunks))}/{len(chunks)}청크 완료", flush=True)
    dense_index[company] = np.vstack(batches)
    dense_rows.append({'company': company, 'e5_build_seconds': perf_counter()-started,
                       'e5_vectors_MiB': dense_index[company].nbytes/1024**2})
display(pd.DataFrame(dense_rows))
print(f'E5 CPU 로드 {model_load_seconds:.2f}초 | 파라미터 메모리 {model_parameter_MiB:.1f} MiB')

## 6. Weighted RRF와 최소 계산 검사

`score = w_vector/(60 + vector_rank) + (1-w_vector)/(60 + bm25_rank)`.
각 검색의 상위 20개 밖에 있으면 그 검색의 기여는 0입니다. 가중치가 0인 검색은 후보에도 기여하지 않습니다.
동점은 청크 인덱스로 고정합니다. 순위 1부터 계산하며, `10:0`/`0:10`도 상위 20개 조건을 동일하게 적용합니다.

In [ ]:
def weighted_rrf(dense_order, lex_order, size, vector_weight, depth=CANDIDATE_DEPTH, k=RRF_K):
    if not 0 <= vector_weight <= 1 or depth < 1 or k < 0:
        raise ValueError('가중치·후보 수·RRF 상수 범위를 확인하세요.')
    scores = np.zeros(size, dtype=np.float64)
    for order, weight in ((dense_order, vector_weight), (lex_order, 1-vector_weight)):
        if weight == 0:
            continue
        ids = np.asarray(order[:depth], dtype=int)
        if len(ids) != len(set(ids)) or (ids < 0).any() or (ids >= size).any():
            raise ValueError('검색 후보 인덱스의 중복·범위를 확인하세요.')
        scores[ids] += weight/(k + np.arange(1, len(ids)+1))
    order = np.argsort(-scores, kind='stable')
    return order[scores[order] > 0]

def score_order(order, relevant, k=EVAL_K):
    first = next((rank for rank, idx in enumerate(order, 1) if int(idx) in relevant), None)
    return {'rank': first, 'hit_at_3': first is not None and first <= 3,
            'hit_at_5': first is not None and first <= k,
            'rr_at_5': 1/first if first is not None and first <= k else 0.0,
            'recall_at_5': len(set(map(int, order[:k])) & relevant)/len(relevant)}

assert list(weighted_rrf([1,0], [0,1], 2, 0.7)) == [1,0]
assert list(weighted_rrf([1,0], [0,1], 2, 0.3)) == [0,1]
assert list(weighted_rrf([1], [0], 2, 1.0)) == [1]
assert list(weighted_rrf([1], [0], 2, 0.0)) == [0]
assert list(weighted_rrf([2,1,0], [], 3, 1.0, depth=2)) == [2,1]
assert list(weighted_rrf([], [], 3, 0.5)) == []
assert score_order([2,1,0], {1,0})['rr_at_5'] == 0.5
assert score_order([2,1], {1,0})['recall_at_5'] == 0.5
assert score_order([], {0})['hit_at_5'] is False
print('RRF 가중치·0가중치·후보 수·빈 후보·순위·Recall 최소 검사 완료')

## 7. 같은 질문에서 다섯 비율 비교

각 질문마다 E5 질문 임베딩, 벡터 순위, BM25 순위를 **한 번만** 계산해 공유합니다.
가중치 비교에서는 검색 순위를 결합하는 연산만 반복합니다.
`full_dense_gold_rank`/`full_bm25_gold_rank`는 후보 20개로 자르기 전 진단용 순위입니다.
정답 인용문은 이 셀의 검색 함수에 전달하지 않고, 순위 산출 뒤 평가에만 사용합니다.

In [ ]:
rows, diagnostics = [], []
for number, case in enumerate(CASES, 1):
    company, question = case['company'], case['question']
    relevant = gold_indices[case['id']]
    print(f"[{number}/{len(CASES)}] {case['id']} {company} 검색 시작", flush=True)
    started = perf_counter()
    lex_order = bm25_order(company, question)
    bm25_ms = (perf_counter()-started)*1000
    started = perf_counter()
    query_vector = encode_texts([question], 'query')[0]
    dense_order = np.argsort(-(dense_index[company] @ query_vector), kind='stable')
    dense_ms = (perf_counter()-started)*1000
    dense_top = set(map(int, dense_order[:CANDIDATE_DEPTH]))
    lex_top = set(map(int, lex_order[:CANDIDATE_DEPTH]))
    diagnostics.append({'id': case['id'], 'company': company,
                        'full_dense_gold_rank': score_order(dense_order, relevant)['rank'],
                        'full_bm25_gold_rank': score_order(lex_order, relevant)['rank'],
                        'gold_in_vector_top20': bool(dense_top & relevant),
                        'gold_in_bm25_top20': bool(lex_top & relevant),
                        'gold_in_union_top20': bool((dense_top|lex_top) & relevant)})
    for ratio, weight in WEIGHTS.items():
        started = perf_counter()
        order = weighted_rrf(dense_order, lex_order, len(chunks_by_company[company]), weight)
        fusion_ms = (perf_counter()-started)*1000
        measured_query_ms = fusion_ms + (dense_ms if weight > 0 else 0) + (bm25_ms if weight < 1 else 0)
        top = list(map(int, order[:EVAL_K]))
        rows.append({'id': case['id'], 'company': company, 'topic': case['topic'],
                     'ratio_vector_bm25': ratio, 'vector_weight': weight,
                     **score_order(order, relevant), 'ranked_count': len(order),
                     'fusion_ms': fusion_ms, 'query_ms': measured_query_ms,
                     'top5_indices': top,
                     'top1_excerpt': chunks_by_company[company][top[0]][:240] if top else ''})
    print('  비율별 근거 순위: ' + ' | '.join(f"{row['ratio_vector_bm25']}={row['rank']}" for row in rows[-5:]), flush=True)
results = pd.DataFrame(rows)
diagnostic_df = pd.DataFrame(diagnostics)
assert len(results) == 125 and results.groupby('ratio_vector_bm25').size().eq(25).all()
display(results[['id', 'company', 'ratio_vector_bm25', 'rank', 'hit_at_5', 'rr_at_5', 'recall_at_5']])

## 8. 전체·기업·질문 유형별 결과와 실패 원인

- **Hit@5:** 정답 인용문을 포함한 청크를 하나라도 상위 5개에 넣은 질문 비율.
- **MRR@5:** 첫 정답 순위의 역수 평균. 상위 5개 밖이면 0.
- **Recall@5:** 각 질문의 알려진 정답 청크 중 상위 5개에 포함한 비율의 평균.
  overlap으로 같은 인용문이 여러 청크에 있으면 Hit와 달라집니다. 전체 정답 의미 근거의 Recall은 아닙니다.
- **기업 macro 평균:** 기업별 지표를 계산한 뒤 기업에 동일한 가중치를 줍니다.
  이번에는 기업당 5질문이므로 전체 평균과 일치해야 합니다.

품질 지표가 엇갈리면 자동 채택하지 않습니다. 양쪽 top20에 근거가 없다면 가중치 변경만으로는 복구할 수 없습니다.

In [ ]:
summary = results.groupby('ratio_vector_bm25').agg(
    questions=('id','count'), hit_at_3=('hit_at_3','mean'), hit_at_5=('hit_at_5','mean'),
    mrr_at_5=('rr_at_5','mean'), recall_at_5=('recall_at_5','mean'),
    median_query_ms=('query_ms','median'), p95_query_ms=('query_ms',lambda s:s.quantile(0.95)),
    median_fusion_ms=('fusion_ms','median')).reindex(WEIGHTS)
by_company = results.groupby(['ratio_vector_bm25','company']).agg(
    questions=('id','count'), hit_at_5=('hit_at_5','mean'), mrr_at_5=('rr_at_5','mean'),
    recall_at_5=('recall_at_5','mean'))
macro = by_company.groupby(level=0)[['hit_at_5','mrr_at_5','recall_at_5']].mean().reindex(WEIGHTS)
by_topic = results.groupby(['ratio_vector_bm25','topic']).agg(
    questions=('id','count'), hit_at_5=('hit_at_5','mean'), mrr_at_5=('rr_at_5','mean'))
display(summary)
print('기업별 결과'); display(by_company)
print('기업 macro 평균'); display(macro)
print('질문 유형별 결과'); display(by_topic)
display(results.pivot(index=['company','id'], columns='ratio_vector_bm25', values='rank').reindex(columns=WEIGHTS))
print('가중치 조정으로 회복 불가능한 후보 누락 문항')
display(diagnostic_df[~diagnostic_df['gold_in_union_top20']])
print('원문 질문·정답과 실패 top1을 함께 확인')
failure_view = results[~results['hit_at_5']].merge(pd.DataFrame(CASES)[['id','question','quote']], on='id')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', 125):
    display(failure_view[['id','ratio_vector_bm25','question','quote','rank','top1_excerpt']])

## 9. 재현·비용 기록과 판단

평가 SHA256, 원문 접수번호·원문 해시, 재사용 함수 해시, 실제 패키지 버전, 모델 revision과
청킹·후보 수·가중치를 출력합니다. 이 정보와 결과를 notebook에 저장합니다.
API 키나 `.env` 내용은 포함하지 않습니다. 별도 평가 산출물 폴더를 새로 만들지 않습니다.

`query_ms`는 공유한 검색 작업의 실측 시간을 후보별 필요한 작업에 더한 **구성요소 시간**입니다.
독립된 운영 요청을 반복 측정한 end-to-end p50/p95가 아니며 작은 표본의 속도 차이는 부차적으로 해석합니다.
파라미터·벡터·SQLite 크기도 프로세스 RAM peak나 전체 원본 보관 비용은 아닙니다.

In [ ]:
manifest = {
    'observed_at_utc': datetime.now(timezone.utc).isoformat(), 'dataset_type':'development',
    'companies':5, 'reports':5, 'questions':25, 'business_year':2024,
    'evaluation_sha256':evaluation_sha256, 'reused_function_sha256':reused_function_sha256,
    'embedding_model':MODEL_ID, 'model_revision':MODEL_REVISION, 'device':'cpu',
    'chunk_size_e5_tokens':400, 'chunk_overlap':48, 'candidate_depth_each':CANDIDATE_DEPTH,
    'rrf_k':RRF_K, 'weights_vector':WEIGHTS, 'model_parameter_MiB':model_parameter_MiB,
    'model_load_seconds':model_load_seconds, 'llm_calls':0, 'openai_embedding_calls':0,
    'packages':{name:importlib.metadata.version(name) for name in
                ['numpy','pandas','torch','transformers','kiwipiepy','llama-index-core','beautifulsoup4']},
}
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print('외부 LLM/임베딩 API 과금 없음. 로컬 CPU·RAM·전력 및 OpenDART 사용 제한은 별도입니다.')
print('기록된 크기 합계: XML %.2f MiB | 텍스트 UTF-8 %.2f MiB | E5 벡터 %.2f MiB | SQLite %.2f MiB' % (
    sum(r['xml_MB'] for r in source_rows), sum(r['text_utf8_MB'] for r in source_rows),
    sum(r['e5_vectors_MiB'] for r in dense_rows), sum(r['bm25_sqlite_MiB'] for r in index_rows)))

### 결과 해석과 남은 한계

1. `7:3` 등 가중치를 조정한 하이브리드가 `10:0`보다 Hit·MRR 모두 우세한지 먼저 봅니다.
   `5:5`보다만 좋아졌다는 이유로 벡터 단독을 미채택하지 않습니다.
2. 지표가 엇갈리면 사용자가 실제 실패 문항·기업별 결과를 본 뒤 보류/선택합니다. 이 notebook은 자동 채택하지 않습니다.
3. 후보 20개 밖의 누락은 비율 문제가 아닐 수 있습니다. `gold_in_union_top20`으로 분리합니다.
4. 삼성전자·리노공업 문항은 이미 본 개발 문항입니다. 새 기업 성능도 기업당 5질문뿐이며,
   원문을 본 작성자의 질문 편향, 제한된 인용문 정답, 단일 연도·산업 분포 한계가 남습니다.
   이번 확대를 독립 holdout이나 최종 운영 성능으로 표현하지 않습니다.
5. 최종 검색 후보는 **사용자와 결과를 검토한 뒤** 결정하고, 가중치를 고르는 데 쓰지 않은 회사/연도/질문으로 확인해야 합니다.
   E5와 OpenAI의 최종 모델 우열, 문자+벡터·BM25+벡터 최종 우열, parent-child 청킹은 이 실험으로 확정하지 않습니다.
6. 저장된 출력을 확인한 뒤 `docs/v1/reports/29_rrf_weight_comparison.html`에 목적·요약·상세 결과·채택/미채택 이유를 작성합니다.
   실행 전 예상 결과 보고서는 만들지 않습니다.

### 근거 자료 (2026-10-05 확인)

- [LangChain Weighted RRF 구현](https://github.com/langchain-ai/langchain/blob/master/libs/langchain/langchain_classic/retrievers/ensemble.py): 순위에 검색기별 가중치를 적용하는 참고. 현재 코드는 LangChain EnsembleRetriever 호출이 아닌 실험용 함수입니다.
- [Elastic RRF](https://www.elastic.co/docs/reference/elasticsearch/rest-apis/reciprocal-rank-fusion): rank constant와 rank window는 결합 가중치와 별도입니다.
- [OpenDART 공시검색](https://opendart.fss.or.kr/guide/detail.do?apiGrpCd=DS001&apiId=2019001): 제출일·기업·A001로 보고서 선택.
- [NAVER 공식 IR 사업보고서](https://www.navercorp.com/investment/irReports): 2024 사업보고서 본문 18·24·32페이지에서 N01~N05 근거 수집.
- [LG생활건강 공식 사업보고서](https://www.lghnh.com/ir/business_report.jsp): 2024 사업보고서 본문 17·29페이지에서 L01~L05 근거 수집.
- [HDC 공식 2024 사업보고서](https://www.hdc-holdings.com/ko/ir/disclosure/notice/view?fromNotice=true&inIdx=6): 본문 14·15·16페이지에서 H01~H05 근거 수집.

공식 IR PDF는 정답 수집의 출처이고 검색 입력은 OpenDART XML에서 추출합니다. XML 안에도 같은 근거가 있는지 4절에서 검사합니다.